[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/23.%20The%20AGV%20Dispatching%20%26%20Battery%20Management%20Problem/P23-Tier-4_executed.ipynb)

---



# Problem 23: The AGV Dispatching and Battery Management Problem

## Tier 4: Deep Reinforcement Learning for Dynamic AGV Dispatching

### Goal and inputs

This notebook builds a small stochastic simulation and trains a Deep Q-Network (DQN) agent to choose dispatching actions. The example uses four automated guided vehicles (AGVs), dynamically arriving transport tasks, battery constraints, and charging stations. The inputs are the terminal locations, travel-time and energy tables, AGV specifications, and charging-station specifications. The outputs are training metrics, test-episode metrics, visualizations, and an analysis of the actions selected by the learned policy.

The aim is to demonstrate the DQN training process—not to claim that the resulting policy is globally optimal. Reinforcement learning uses trial-and-error interaction with an environment; a policy's quality depends on the model, reward design, training experience, and evaluation conditions.

### Scope and modeling assumptions

- The simulation represents task completion probabilistically rather than modeling vehicle motion, routes, collision avoidance, or detailed service durations. Travel times are calculated as input data, but this compact environment does not use them to advance the simulation clock.
- Battery feasibility is checked using energy estimates for travel from the AGV's current location to pickup, from pickup to drop-off, and from drop-off to the depot. Upon task completion, the battery is reduced by the energy for the first two legs; the depot leg is retained as a safety reserve.
- Charging is an abstract activity: choosing a station applies that station's charging rate without modeling travel to the station, charger capacity, or queueing. Station queue lengths are included in the state representation but are not dynamically updated.
- New tasks arrive stochastically, and a moving AGV completes its assigned task with a fixed probability at each simulation step. These assumptions make the example suitable for illustrating the DQN workflow, not for operational planning.
- The action is one AGV and one action type per time step. Invalid or infeasible choices receive penalties. The task, charging, and idle action slots have fixed indices, so an action keeps the same meaning when the pending-task list changes.

### DQN in this solution method

At each step, the environment provides a state vector $s_t$. The agent selects an action $a_t$, receives reward $r_t$, and observes the next state $s_{t+1}$. A neural network approximates the action-value function $Q(s,a)$: the expected discounted return from taking action $a$ in state $s$ and then following the policy. During training, an epsilon-greedy rule sometimes explores by choosing a random action; otherwise, it chooses an action with the largest estimated Q-value.

For a sampled transition, the DQN target is

$$y_t = r_t + \gamma(1-d_t)\max_{a'} Q_{\text{target}}(s_{t+1},a'),$$

and the network is trained to reduce the squared temporal-difference error

$$L(\theta)=\frac{1}{B}\sum_{i=1}^{B}\left(Q_{\theta}(s_i,a_i)-y_i\right)^2.$$

Here, $d_t$ is 1 when the transition ends an episode and 0 otherwise; $\gamma$ discounts future rewards; $Q_{\text{target}}$ is the periodically synchronized target network; $B$ is the replay minibatch size; and $\theta$ denotes the trainable network parameters. Experience replay stores past transitions and samples them in random minibatches, reducing the dependence between consecutive training examples. The target network supplies a temporarily stable target while the online network learns. In the code, these ideas appear in `DQNAgent.remember`, `DQNAgent.replay`, and `DQNAgent.update_target_network`.

The state includes normalized AGV battery, location, and status; a fixed number of pending-task features; station queue lengths; and global time and task-count features. The action space assigns each AGV a fixed block containing one slot per possible task, one per charging station, and one idle slot. The network's output has one Q-value per encoded action.

### Workflow

1. Define the simulation data structures and environment.
2. Encode the state and fixed action space.
3. Define the DQN and its replay-based learning rule.
4. Create the example terminal, AGVs, and charging stations.
5. Train the agent, evaluate it without exploratory actions, and inspect its behavior.

Training curves and test results are empirical observations from the run. They do not, by themselves, establish convergence, optimality, or statistical superiority over another policy.

In [ ]:
from collections import defaultdict, deque, namedtuple
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch is unavailable; the demonstration will use its NumPy implementation.")

# Fixed seeds make the example reproducible within a given software environment.
np.random.seed(42)
random.seed(42)
if TORCH_AVAILABLE:
    torch.manual_seed(42)

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Task:
    """Transport task with locations, timing information, and priority."""

    id: str
    pickup: str
    dropoff: str
    time_window: Tuple[float, float]
    service_time: float
    priority: float = 1.0
    deadline: float = float("inf")
    arrival_time: float = 0.0


@dataclass
class AGV:
    """AGV specifications and its current simulated operating state."""

    id: str
    battery_capacity: float
    battery_min: float
    initial_battery: float
    speed: float = 1.0
    current_location: str = "O"
    current_battery: float = 100.0
    current_task: Optional[str] = None
    status: str = "idle"


@dataclass
class ChargingStation:
    """Charging rate and queue indicator for a charging station."""

    id: str
    charging_rate: float
    queue_length: int = 0


@dataclass
class Location:
    """Two-dimensional terminal location."""

    id: str
    x: float
    y: float

    def distance_to(self, other: "Location") -> float:
        """Return Euclidean distance to another location."""
        return float(np.hypot(self.x - other.x, self.y - other.y))


Experience = namedtuple(
    "Experience", ["state", "action", "reward", "next_state", "done"]
)


@dataclass
class DRLMetrics:
    """Training measurements recorded across episodes and updates."""

    episode_rewards: List[float] = field(default_factory=list)
    episode_lengths: List[int] = field(default_factory=list)
    loss_history: List[float] = field(default_factory=list)
    epsilon_history: List[float] = field(default_factory=list)
    success_rates: List[float] = field(default_factory=list)
    battery_violations: List[int] = field(default_factory=list)

In [ ]:
class TaskGenerator:
    """Generate random pickup-and-drop-off tasks from the supplied locations."""

    def __init__(self, locations):
        self.locations = locations
        self.task_counter = 0
        self.pickup_nodes = [
            loc_id for loc_id in locations if loc_id.startswith("P")
        ]
        self.dropoff_nodes = [
            loc_id for loc_id in locations if loc_id.startswith("D")
        ]
        if not self.pickup_nodes or not self.dropoff_nodes:
            raise ValueError("At least one pickup and one drop-off location are required.")

    def generate_task(self, current_time):
        """Create one task and stamp its arrival time."""
        self.task_counter += 1
        return Task(
            id=f"T{self.task_counter}",
            pickup=random.choice(self.pickup_nodes),
            dropoff=random.choice(self.dropoff_nodes),
            time_window=(current_time, current_time + random.uniform(20, 40)),
            service_time=random.uniform(1, 3),
            priority=random.uniform(0.5, 1.5),
            deadline=current_time + random.uniform(30, 60),
            arrival_time=current_time,
        )


class AGVDispatchingEnvironment:
    """Stochastic AGV dispatching environment for DQN training."""

    def __init__(
        self,
        locations,
        travel_times,
        energy_consumption,
        charging_stations,
        agvs,
        max_tasks=10,
    ):
        if max_tasks <= 0:
            raise ValueError("max_tasks must be positive.")
        if not charging_stations:
            raise ValueError("At least one charging station is required.")

        self.locations = locations
        self.travel_times = travel_times
        self.energy_consumption = energy_consumption
        self.charging_stations = charging_stations
        self.agv_specs = agvs
        self.max_tasks = max_tasks
        self.num_agvs = len(agvs)
        self.num_stations = len(charging_stations)
        self.action_stride = self.max_tasks + self.num_stations + 1
        self.state_size = self.calculate_state_size()
        self.action_size = self.calculate_action_size()
        self.max_episode_steps = 100
        self.task_generator = TaskGenerator(locations)
        self.reset()

    def calculate_state_size(self):
        """Count the fixed-width state features used by the network."""
        agv_state_size = self.num_agvs * 3
        task_state_size = self.max_tasks * 4
        station_state_size = self.num_stations
        global_state_size = 2
        return (
            agv_state_size
            + task_state_size
            + station_state_size
            + global_state_size
        )

    def calculate_action_size(self):
        """Count one fixed block of task, charging, and idle choices per AGV."""
        return self.num_agvs * self.action_stride

    def reset_agvs(self):
        """Restore each AGV to its specified initial battery and depot location."""
        self.agvs = [
            AGV(
                id=spec.id,
                battery_capacity=spec.battery_capacity,
                battery_min=spec.battery_min,
                initial_battery=spec.initial_battery,
                speed=spec.speed,
                current_location="O",
                current_battery=spec.initial_battery,
                status="idle",
            )
            for spec in self.agv_specs
        ]

    def reset(self):
        """Start a fresh episode and return its initial state vector."""
        self.current_time = 0.0
        self.episode_step = 0
        self.total_reward = 0.0
        self.battery_violations = 0
        self.tasks_completed = 0
        self.reset_agvs()
        self.pending_tasks = []
        self.completed_tasks = []
        self.assigned_tasks = {}
        self.agv_station = {}

        for _ in range(min(3, self.max_tasks)):
            self.pending_tasks.append(
                self.task_generator.generate_task(self.current_time)
            )
        return self.get_state()

    def get_state(self):
        """Encode AGVs, pending tasks, stations, and global context as a vector."""
        state = []
        location_denominator = max(len(self.locations) - 1, 1)

        for agv in self.agvs:
            state.extend(
                [
                    agv.current_battery / agv.battery_capacity,
                    self.location_to_index(agv.current_location)
                    / location_denominator,
                    self.status_to_index(agv.status) / 3.0,
                ]
            )

        for task_index in range(self.max_tasks):
            if task_index < len(self.pending_tasks):
                task = self.pending_tasks[task_index]
                remaining_deadline = max(task.deadline - self.current_time, 0.0)
                state.extend(
                    [
                        self.location_to_index(task.pickup) / location_denominator,
                        self.location_to_index(task.dropoff) / location_denominator,
                        task.priority / 2.0,
                        min(remaining_deadline / 100.0, 1.0),
                    ]
                )
            else:
                state.extend([0.0, 0.0, 0.0, 0.0])

        for station in self.charging_stations:
            state.append(min(station.queue_length / 5.0, 1.0))

        state.extend(
            [
                min(self.current_time / 100.0, 1.0),
                min(len(self.pending_tasks) / self.max_tasks, 1.0),
            ]
        )
        state_array = np.asarray(state, dtype=np.float32)
        if state_array.size != self.state_size:
            raise RuntimeError("The encoded state does not match state_size.")
        return state_array

    def location_to_index(self, location_id):
        """Map a location identifier to its stable index in the input mapping."""
        location_map = {
            loc_id: index for index, loc_id in enumerate(self.locations)
        }
        return location_map.get(location_id, 0)

    @staticmethod
    def status_to_index(status):
        """Map the four modeled AGV statuses to numeric state features."""
        status_map = {"idle": 0, "moving": 1, "charging": 2, "servicing": 3}
        return status_map.get(status, 0)

    def decode_action(self, action):
        """Decode a flat action index into an AGV and its fixed action slot."""
        if not 0 <= action < self.action_size:
            raise ValueError("Action index is outside the defined action space.")
        agv_index = action // self.action_stride
        action_type = action % self.action_stride
        return agv_index, action_type

    def step(self, action):
        """Apply one dispatch decision and return state, reward, terminal flag, and info."""
        self.episode_step += 1
        agv_index, action_type = self.decode_action(action)
        reward = self.execute_action(agv_index, action_type)

        self.update_environment()
        if (
            random.random() < 0.1
            and len(self.pending_tasks) < self.max_tasks
        ):
            self.pending_tasks.append(
                self.task_generator.generate_task(self.current_time)
            )

        done = self.is_episode_done()
        self.total_reward += reward
        return self.get_state(), reward, done, self.get_info()

    def execute_action(self, agv_index, action_type):
        """Apply a task assignment, charging choice, or idle choice and score it."""
        agv = self.agvs[agv_index]

        if action_type < self.max_tasks:
            if action_type >= len(self.pending_tasks):
                return -1.0
            if agv.status != "idle":
                return -1.0

            task = self.pending_tasks[action_type]
            if not self.can_handle_task(agv, task):
                return -2.0

            agv.current_task = task.id
            agv.status = "moving"
            self.assigned_tasks[agv.id] = task
            self.pending_tasks.pop(action_type)
            return 5.0

        charge_end = self.max_tasks + self.num_stations
        if action_type < charge_end:
            station_index = action_type - self.max_tasks
            if agv.status != "idle":
                return -1.0
            if agv.current_battery >= agv.battery_capacity * 0.3:
                return -0.5

            agv.status = "charging"
            self.agv_station[agv.id] = station_index
            return 2.0

        if agv.status == "idle" and self.pending_tasks:
            return -1.0
        return 0.0

    def can_handle_task(self, agv, task):
        """Check that task energy plus a return-to-depot reserve is available."""
        energy_to_pickup = self.energy_consumption.get(
            (agv.current_location, task.pickup), 0.0
        )
        energy_for_task = self.energy_consumption.get(
            (task.pickup, task.dropoff), 0.0
        )
        energy_to_depot = self.energy_consumption.get((task.dropoff, "O"), 0.0)
        required_energy = energy_to_pickup + energy_for_task + energy_to_depot
        return agv.current_battery >= required_energy + agv.battery_min

    def update_environment(self):
        """Advance simulated time and update task completion and charging."""
        self.current_time += 1.0

        for agv in self.agvs:
            if agv.status == "moving" and agv.current_task is not None:
                if random.random() < 0.3:
                    task = self.assigned_tasks.pop(agv.id)
                    energy_to_pickup = self.energy_consumption.get(
                        (agv.current_location, task.pickup), 0.0
                    )
                    energy_for_task = self.energy_consumption.get(
                        (task.pickup, task.dropoff), 0.0
                    )
                    agv.current_battery -= energy_to_pickup + energy_for_task
                    agv.current_location = task.dropoff
                    agv.current_task = None
                    agv.status = "idle"
                    self.completed_tasks.append(task)
                    self.tasks_completed += 1

                    if agv.current_battery < agv.battery_min:
                        self.battery_violations += 1
                        agv.current_battery = agv.battery_min

            elif agv.status == "charging":
                station_index = self.agv_station[agv.id]
                station = self.charging_stations[station_index]
                agv.current_battery = min(
                    agv.current_battery + station.charging_rate,
                    agv.battery_capacity,
                )
                if agv.current_battery >= agv.battery_capacity * 0.9:
                    agv.status = "idle"
                    self.agv_station.pop(agv.id, None)

    def is_episode_done(self):
        """End on the step limit, task target, or a fully idle empty system."""
        no_work_remains = not self.pending_tasks and all(
            agv.status == "idle" for agv in self.agvs
        )
        return (
            self.episode_step >= self.max_episode_steps
            or self.tasks_completed >= self.max_tasks
            or no_work_remains
        )

    def get_info(self):
        """Return episode measurements used by training and evaluation."""
        return {
            "time": self.current_time,
            "tasks_completed": self.tasks_completed,
            "battery_violations": self.battery_violations,
            "pending_tasks": len(self.pending_tasks),
            "total_reward": self.total_reward,
        }

## State, action, and reward design

The state vector has a fixed length because a neural network expects the same number of input features at every step. For each AGV it stores battery fraction, normalized location index, and normalized status. Each task slot stores pickup and drop-off indices, priority, and remaining-deadline feature; unused slots are zero-padded. Station queue features and two global features complete the vector. The resulting state is a compact representation, not a complete description of every physical detail.

The action index is organized into equal-sized blocks, one per AGV. Within each block, slots $0,\ldots,M-1$ refer to task positions, the next $C$ slots refer to charging stations, and the final slot means idle; $M$ is `max_tasks` and $C$ is `num_stations`. Thus, the action space has size $N_{\mathrm{AGV}}(M+C+1)$. Task slots beyond the number of currently pending tasks are invalid and receive a penalty. Fixed slots are important: the meaning of a charging or idle action does not shift as tasks enter or leave the queue.

Rewards provide the learning signal: feasible task assignment is rewarded, infeasible or unavailable choices are penalized, and charging is rewarded only below the modeled battery threshold. This reward design shapes behavior but does not guarantee that maximizing the simulated reward is equivalent to optimizing every real-world operational objective.

In [ ]:
if TORCH_AVAILABLE:
    class DQN(nn.Module):
        """Fully connected neural network that estimates one Q-value per action."""

        def __init__(self, state_size, action_size, hidden_sizes=(128, 64, 32)):
            super().__init__()
            layers = []
            input_size = state_size
            for hidden_size in hidden_sizes:
                layers.extend([nn.Linear(input_size, hidden_size), nn.ReLU()])
                input_size = hidden_size
            layers.append(nn.Linear(input_size, action_size))
            self.network = nn.Sequential(*layers)

        def forward(self, state):
            return self.network(state)
else:
    class SimpleNeuralNetwork:
        """Two-layer NumPy network with a ReLU hidden layer."""

        def __init__(self, state_size, action_size, hidden_size=64):
            self.state_size = state_size
            self.action_size = action_size
            self.hidden_size = hidden_size
            self.W1 = np.random.randn(state_size, hidden_size) * 0.1
            self.W2 = np.random.randn(hidden_size, action_size) * 0.1
            self.b1 = np.zeros(hidden_size)
            self.b2 = np.zeros(action_size)

        def forward(self, state):
            hidden = np.maximum(0.0, np.dot(state, self.W1) + self.b1)
            return np.dot(hidden, self.W2) + self.b2

        def __call__(self, state):
            return self.forward(state)

    DQN = SimpleNeuralNetwork


class DQNAgent:
    """DQN learner using epsilon-greedy actions, replay, and a target network."""

    def __init__(self, state_size, action_size, lr=0.001):
        self.state_size = state_size
        self.action_size = action_size
        self.lr = lr

        self.q_network = DQN(state_size, action_size)
        self.target_network = DQN(state_size, action_size)
        if TORCH_AVAILABLE:
            self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
            self.loss_fn = nn.MSELoss()
            self.target_network.load_state_dict(self.q_network.state_dict())
            self.target_network.eval()
        else:
            self.copy_numpy_weights()

        self.memory = deque(maxlen=10000)
        self.batch_size = 32
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.gamma = 0.95
        self.update_target_every = 10
        self.metrics = DRLMetrics()

    def copy_numpy_weights(self):
        """Synchronize the NumPy target network with the online network."""
        self.target_network.W1 = self.q_network.W1.copy()
        self.target_network.W2 = self.q_network.W2.copy()
        self.target_network.b1 = self.q_network.b1.copy()
        self.target_network.b2 = self.q_network.b2.copy()

    def remember(self, state, action, reward, next_state, done):
        """Save a transition so learning can use randomized past experience."""
        self.memory.append(
            Experience(
                np.asarray(state, dtype=np.float32).copy(),
                int(action),
                float(reward),
                np.asarray(next_state, dtype=np.float32).copy(),
                bool(done),
            )
        )

    def act(self, state, training=True):
        """Select a random exploratory action or the highest-valued action."""
        if training and np.random.random() <= self.epsilon:
            return random.randrange(self.action_size)

        if TORCH_AVAILABLE:
            self.q_network.eval()
            with torch.no_grad():
                state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
                q_values = self.q_network(state_tensor)
            return int(torch.argmax(q_values, dim=1).item())

        return int(np.argmax(self.q_network(state)))

    def replay(self):
        """Update the online Q-network from a random minibatch of transitions."""
        if len(self.memory) < self.batch_size:
            return

        batch = random.sample(self.memory, self.batch_size)
        if TORCH_AVAILABLE:
            states = torch.as_tensor(
                np.stack([item.state for item in batch]), dtype=torch.float32
            )
            actions = torch.as_tensor(
                [item.action for item in batch], dtype=torch.long
            )
            rewards = torch.as_tensor(
                [item.reward for item in batch], dtype=torch.float32
            )
            next_states = torch.as_tensor(
                np.stack([item.next_state for item in batch]), dtype=torch.float32
            )
            dones = torch.as_tensor(
                [item.done for item in batch], dtype=torch.float32
            )

            self.q_network.train()
            current_q_values = self.q_network(states).gather(
                1, actions.unsqueeze(1)
            ).squeeze(1)
            self.target_network.eval()
            with torch.no_grad():
                next_q_values = self.target_network(next_states).max(dim=1).values
                targets = rewards + self.gamma * (1.0 - dones) * next_q_values

            loss = self.loss_fn(current_q_values, targets)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.metrics.loss_history.append(float(loss.item()))
            return

        for experience in batch:
            current_values = self.q_network(experience.state)
            current_q = current_values[experience.action]
            if experience.done:
                target_q = experience.reward
            else:
                target_q = experience.reward + self.gamma * np.max(
                    self.target_network(experience.next_state)
                )
            td_error = target_q - current_q
            self.metrics.loss_history.append(float(0.5 * td_error**2))
            self.simple_update(experience.state, experience.action, td_error)

    def simple_update(self, state, action, td_error):
        """Apply back-propagation for the selected action in the NumPy network."""
        learning_rate = self.lr * 0.01
        pre_activation = np.dot(state, self.q_network.W1) + self.q_network.b1
        hidden = np.maximum(0.0, pre_activation)
        output_weights = self.q_network.W2[:, action].copy()

        self.q_network.W2[:, action] += learning_rate * td_error * hidden
        self.q_network.b2[action] += learning_rate * td_error

        hidden_gradient = td_error * output_weights * (pre_activation > 0.0)
        self.q_network.W1 += learning_rate * np.outer(state, hidden_gradient)
        self.q_network.b1 += learning_rate * hidden_gradient

    def update_target_network(self):
        """Copy online parameters to the target network at the scheduled interval."""
        if TORCH_AVAILABLE:
            self.target_network.load_state_dict(self.q_network.state_dict())
            self.target_network.eval()
        else:
            self.copy_numpy_weights()

    def train(self, env, episodes=100):
        """Train over complete episodes and return the accumulated measurements."""
        print(f"Training the DQN agent for {episodes} episodes...")

        for episode in range(episodes):
            state = env.reset()
            total_reward = 0.0
            steps = 0
            done = False
            info = env.get_info()

            while not done:
                action = self.act(state)
                next_state, reward, done, info = env.step(action)
                self.remember(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward
                steps += 1

            self.replay()
            self.epsilon = max(
                self.epsilon_min, self.epsilon * self.epsilon_decay
            )

            if (episode + 1) % self.update_target_every == 0:
                self.update_target_network()

            self.metrics.episode_rewards.append(total_reward)
            self.metrics.episode_lengths.append(steps)
            self.metrics.epsilon_history.append(self.epsilon)
            self.metrics.battery_violations.append(info["battery_violations"])
            self.metrics.success_rates.append(
                info["tasks_completed"] / env.max_tasks
            )

            if (episode + 1) % 20 == 0:
                average_reward = np.mean(self.metrics.episode_rewards[-20:])
                print(
                    f"  Episode {episode + 1:3d}: "
                    f"mean reward={average_reward:.2f}, "
                    f"epsilon={self.epsilon:.3f}, "
                    f"success fraction={self.metrics.success_rates[-1]:.2f}"
                )

        print("Training complete.")
        return self.metrics

## Example terminal and fleet

The following coordinates define the example geometry. Pairwise Euclidean distance is used to construct travel-time and energy tables: travel time is distance divided by the reference speed, and energy use is distance multiplied by the energy-per-distance factor. AGV-specific speed is retained in the specifications, while this simplified environment uses the shared tables for battery feasibility.

In [ ]:
drl_locations = {
    "O": Location("O", 0, 0),
    "P1": Location("P1", 2, 3),
    "D1": Location("D1", 6, 4),
    "P2": Location("P2", 1, 5),
    "D2": Location("D2", 5, 7),
    "P3": Location("P3", 3, 1),
    "D3": Location("D3", 7, 2),
    "P4": Location("P4", 4, 6),
    "D4": Location("D4", 8, 5),
    "C1": Location("C1", 3, 3),
    "C2": Location("C2", 6, 1),
}


def calculate_drl_matrices(locations):
    """Build complete pairwise travel-time and energy lookup tables."""
    travel_times = {}
    energy_consumption = {}

    for first_id, first_location in locations.items():
        for second_id, second_location in locations.items():
            distance = first_location.distance_to(second_location)
            travel_times[(first_id, second_id)] = distance / 1.0
            energy_consumption[(first_id, second_id)] = distance * 1.2

    return travel_times, energy_consumption


drl_travel_times, drl_energy_consumption = calculate_drl_matrices(drl_locations)

drl_agvs = [
    AGV("K1", 100.0, 20.0, 100.0, speed=1.0),
    AGV("K2", 120.0, 25.0, 120.0, speed=1.2),
    AGV("K3", 90.0, 15.0, 90.0, speed=0.9),
    AGV("K4", 110.0, 22.0, 110.0, speed=1.1),
]

drl_charging_stations = [
    ChargingStation("C1", 8.0),
    ChargingStation("C2", 12.0),
]

print("DRL environment setup:")
print(f"Locations: {len(drl_locations)}")
print(f"AGVs: {len(drl_agvs)}")
print(f"Charging stations: {len(drl_charging_stations)}")
print(f"PyTorch available: {TORCH_AVAILABLE}")

In [ ]:
env = AGVDispatchingEnvironment(
    locations=drl_locations,
    travel_times=drl_travel_times,
    energy_consumption=drl_energy_consumption,
    charging_stations=drl_charging_stations,
    agvs=drl_agvs,
    max_tasks=8,
)

print(f"Environment state size: {env.state_size}")
print(f"Environment action size: {env.action_size}")

agent = DQNAgent(
    state_size=env.state_size,
    action_size=env.action_size,
    lr=0.001,
)

print(f"Agent learning rate: {agent.lr}")
print(f"Initial exploration rate (epsilon): {agent.epsilon}")

initial_state = env.reset()
assert initial_state.shape == (env.state_size,)
assert env.action_size == env.num_agvs * (
    env.max_tasks + env.num_stations + 1
)

## Training

Each episode starts from a reset environment. At each step, the agent acts, the environment returns a transition, and the transition is stored in replay memory. Once the memory contains a full minibatch, `replay` updates the online network. Epsilon decreases after each episode, shifting the action-selection rule gradually from exploration toward exploitation. The target network is synchronized at the configured episode interval. The episode count, learning rate, discount factor, replay capacity, and other settings are kept as explicit example parameters.

In [ ]:
start_time = time.time()
metrics = agent.train(env, episodes=100)
training_time = time.time() - start_time

print(f"\nElapsed training time: {training_time:.2f} seconds")
print(f"Final exploration rate (epsilon): {agent.epsilon:.3f}")
print("\nTraining summary")
print("=" * 60)
if metrics.episode_rewards:
    print(
        "Mean reward over the final 10 episodes: "
        f"{np.mean(metrics.episode_rewards[-10:]):.2f}"
    )
    print(f"Best episode reward: {np.max(metrics.episode_rewards):.2f}")
    print(f"Mean task-completion fraction: {np.mean(metrics.success_rates):.3f}")
    print(f"Mean episode length: {np.mean(metrics.episode_lengths):.1f} steps")
    print(f"Battery violations across training episodes: {sum(metrics.battery_violations)}")
else:
    print("No training metrics were recorded.")

## Evaluation without exploration

Evaluation uses the greedy policy (`training=False`), so epsilon-based random actions are disabled. The simulation itself remains stochastic: task arrivals and completion events can vary between episodes. Mean and standard deviation describe variation across the sampled episodes; they are not confidence intervals and do not establish statistical significance.

In [ ]:
def test_trained_agent(agent, env, num_episodes=10):
    """Evaluate the greedy policy over independently simulated episodes."""
    print("\n" + "=" * 60)
    print("EVALUATING THE TRAINED AGENT")
    print("=" * 60)

    test_rewards = []
    test_success_rates = []
    test_violations = []

    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        info = env.get_info()

        while not done:
            action = agent.act(state, training=False)
            next_state, reward, done, info = env.step(action)
            state = next_state
            total_reward += reward

        success_rate = info["tasks_completed"] / env.max_tasks
        test_rewards.append(total_reward)
        test_success_rates.append(success_rate)
        test_violations.append(info["battery_violations"])

        print(
            f"Test episode {episode + 1:2d}: "
            f"reward={total_reward:7.2f}, "
            f"task-completion fraction={success_rate:.2f}, "
            f"battery violations={info['battery_violations']}"
        )

    print("\nAggregate test results")
    print(
        f"Mean reward: {np.mean(test_rewards):.2f} "
        f"(standard deviation {np.std(test_rewards):.2f})"
    )
    print(
        f"Mean task-completion fraction: {np.mean(test_success_rates):.3f} "
        f"(standard deviation {np.std(test_success_rates):.3f})"
    )
    print(f"Battery violations across test episodes: {sum(test_violations)}")

    return {
        "rewards": test_rewards,
        "success_rates": test_success_rates,
        "violations": test_violations,
    }


test_results = test_trained_agent(agent, env, num_episodes=10)

## Comparison with a simple baseline

The baseline repeatedly selects the first AGV's first task slot. This deliberately simple policy is included as a reference point, not as another optimization method. Because the baseline and DQN evaluations experience stochastic events and are run sequentially rather than on paired random scenarios, observed differences are descriptive only. A percentage reward difference is reported relative to the absolute baseline mean when that mean is nonzero; reward percentages can be difficult to interpret when rewards are negative, so the underlying means should also be inspected.

In [ ]:
def compare_with_baseline(agent, env, num_episodes=5):
    """Report descriptive results for the DQN policy and a fixed simple baseline."""
    print("\n" + "=" * 60)
    print("COMPARISON WITH A SIMPLE BASELINE")
    print("=" * 60)

    drl_results = test_trained_agent(agent, env, num_episodes)
    baseline_rewards = []
    baseline_success_rates = []
    baseline_violations = []

    for _ in range(num_episodes):
        env.reset()
        total_reward = 0.0
        done = False
        info = env.get_info()

        while not done:
            action = 0
            _, reward, done, info = env.step(action)
            total_reward += reward

        baseline_rewards.append(total_reward)
        baseline_success_rates.append(info["tasks_completed"] / env.max_tasks)
        baseline_violations.append(info["battery_violations"])

    drl_average_reward = float(np.mean(drl_results["rewards"]))
    baseline_average_reward = float(np.mean(baseline_rewards))
    if not np.isclose(baseline_average_reward, 0.0):
        reward_improvement = (
            (drl_average_reward - baseline_average_reward)
            / abs(baseline_average_reward)
            * 100.0
        )
    else:
        reward_improvement = float("nan")

    drl_average_success = float(np.mean(drl_results["success_rates"]))
    baseline_average_success = float(np.mean(baseline_success_rates))
    if not np.isclose(baseline_average_success, 0.0):
        success_improvement = (
            (drl_average_success - baseline_average_success)
            / abs(baseline_average_success)
            * 100.0
        )
    else:
        success_improvement = float("nan")

    print("DQN policy:")
    print(f"  Mean reward: {drl_average_reward:.2f}")
    print(f"  Mean task-completion fraction: {drl_average_success:.3f}")
    print(f"  Battery violations: {sum(drl_results['violations'])}")

    print("\nSimple baseline:")
    print(f"  Mean reward: {baseline_average_reward:.2f}")
    print(f"  Mean task-completion fraction: {baseline_average_success:.3f}")
    print(f"  Battery violations: {sum(baseline_violations)}")

    print("\nDescriptive relative differences:")
    print(f"  Reward difference: {reward_improvement:.1f}%")
    print(f"  Task-completion difference: {success_improvement:.1f}%")

    return {
        "drl_reward": drl_average_reward,
        "baseline_reward": baseline_average_reward,
        "drl_success": drl_average_success,
        "baseline_success": baseline_average_success,
        "reward_improvement": reward_improvement,
        "success_improvement": success_improvement,
    }


comparison = compare_with_baseline(agent, env, num_episodes=5)

## Training and performance visualizations

The plots summarize episode rewards, the exploration schedule alongside task-completion fraction, the evaluation means, and the temporal-difference loss recorded during replay updates. A moving average can make broad trends easier to see, but it does not remove stochastic variation. Loss is a training diagnostic, not a direct measure of operational performance.

In [ ]:
def visualize_drl_results(metrics, test_results, comparison):
    """Plot training history, test summaries, and replay loss."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "DQN Training and Evaluation Summary",
        fontsize=14,
        fontweight="bold",
    )

    reward_axis = axes[0, 0]
    if metrics.episode_rewards:
        episodes = np.arange(1, len(metrics.episode_rewards) + 1)
        reward_axis.plot(
            episodes,
            metrics.episode_rewards,
            color="tab:blue",
            alpha=0.35,
            linewidth=1,
            label="Episode reward",
        )
        window = min(10, len(metrics.episode_rewards))
        if window > 1:
            moving_average = (
                pd.Series(metrics.episode_rewards)
                .rolling(window=window, min_periods=1)
                .mean()
            )
            reward_axis.plot(
                episodes,
                moving_average,
                color="tab:blue",
                linewidth=2,
                label=f"Moving average ({window} episodes)",
            )
        reward_axis.set_xlabel("Episode")
        reward_axis.set_ylabel("Total reward (model units)")
        reward_axis.set_title("Training reward")
        reward_axis.grid(True, alpha=0.3)
        reward_axis.legend()

    schedule_axis = axes[0, 1]
    if metrics.epsilon_history:
        episodes = np.arange(1, len(metrics.epsilon_history) + 1)
        success_axis = schedule_axis.twinx()
        epsilon_line = schedule_axis.plot(
            episodes,
            metrics.epsilon_history,
            color="tab:red",
            label="Epsilon (exploration rate)",
        )
        success_line = success_axis.plot(
            episodes,
            metrics.success_rates,
            color="tab:green",
            label="Task-completion fraction",
        )
        schedule_axis.set_xlabel("Episode")
        schedule_axis.set_ylabel("Epsilon", color="tab:red")
        success_axis.set_ylabel("Task-completion fraction", color="tab:green")
        schedule_axis.set_title("Exploration schedule and episode outcomes")
        schedule_axis.grid(True, alpha=0.3)
        lines = epsilon_line + success_line
        schedule_axis.legend(lines, [line.get_label() for line in lines], loc="best")

    comparison_axis = axes[1, 0]
    method_names = ["Simple baseline", "DQN policy"]
    mean_rewards = [comparison["baseline_reward"], comparison["drl_reward"]]
    mean_success = [comparison["baseline_success"], comparison["drl_success"]]
    positions = np.arange(len(method_names))
    width = 0.36
    reward_bars = comparison_axis.bar(
        positions - width / 2,
        mean_rewards,
        width,
        color="lightblue",
        label="Mean reward",
    )
    success_axis = comparison_axis.twinx()
    success_bars = success_axis.bar(
        positions + width / 2,
        mean_success,
        width,
        color="lightgreen",
        label="Mean task-completion fraction",
    )
    comparison_axis.set_xlabel("Policy")
    comparison_axis.set_ylabel("Mean reward (model units)")
    success_axis.set_ylabel("Mean task-completion fraction")
    comparison_axis.set_title("Descriptive evaluation comparison")
    comparison_axis.set_xticks(positions)
    comparison_axis.set_xticklabels(method_names)
    comparison_axis.grid(True, axis="y", alpha=0.3)
    comparison_axis.legend([reward_bars, success_bars], [
        "Mean reward", "Mean task-completion fraction"
    ], loc="best")

    for bar in reward_bars:
        height = bar.get_height()
        comparison_axis.annotate(
            f"{height:.2f}",
            (bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3 if height >= 0 else -12),
            textcoords="offset points",
            ha="center",
        )
    for bar in success_bars:
        height = bar.get_height()
        success_axis.annotate(
            f"{height:.2f}",
            (bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
        )

    loss_axis = axes[1, 1]
    if metrics.loss_history:
        update_steps = np.arange(1, len(metrics.loss_history) + 1)
        loss_axis.plot(
            update_steps,
            metrics.loss_history,
            color="purple",
            alpha=0.55,
            linewidth=1,
            label="Minibatch loss",
        )
        if len(metrics.loss_history) > 10:
            window = min(20, len(metrics.loss_history))
            loss_average = (
                pd.Series(metrics.loss_history)
                .rolling(window=window, min_periods=1)
                .mean()
            )
            loss_axis.plot(
                update_steps,
                loss_average,
                color="tab:red",
                linewidth=2,
                label=f"Moving average ({window} updates)",
            )
        loss_axis.set_xlabel("Replay update")
        loss_axis.set_ylabel("Mean squared TD error")
        loss_axis.set_title("Training loss")
        loss_axis.grid(True, alpha=0.3)
        loss_axis.legend()

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_drl_results(metrics, test_results, comparison)

## Learned-policy action analysis

This analysis follows the greedy action from the current state for a fixed number of steps, resetting the environment whenever an episode ends. It records action counts and immediate rewards by action. The counts describe this sampled trajectory; they do not prove that the policy has learned a general strategy. In particular, immediate rewards do not measure the full long-term return used by Q-learning.

In [ ]:
def analyze_learned_policy(agent, env, num_steps=50):
    """Summarize greedy policy actions and their observed immediate rewards."""
    print("\n" + "=" * 60)
    print("POLICY ACTION ANALYSIS")
    print("=" * 60)

    state = env.reset()
    action_counts = [0] * env.action_size
    reward_by_action = defaultdict(list)

    for _ in range(num_steps):
        action = agent.act(state, training=False)
        next_state, reward, done, _ = env.step(action)
        action_counts[action] += 1
        reward_by_action[action].append(reward)
        state = env.reset() if done else next_state

    task_actions = 0
    charge_actions = 0
    idle_actions = 0
    print(f"\nAction counts over {num_steps} decisions:")

    for action_index, count in enumerate(action_counts):
        if count == 0:
            continue
        agv_index, action_type = env.decode_action(action_index)
        if action_type < env.max_tasks:
            task_actions += count
            action_name = f"AGV {agv_index} -> task slot {action_type}"
        elif action_type < env.max_tasks + env.num_stations:
            charge_actions += count
            station_index = action_type - env.max_tasks
            action_name = f"AGV {agv_index} -> charging station {station_index}"
        else:
            idle_actions += count
            action_name = f"AGV {agv_index} -> idle"

        average_reward = float(np.mean(reward_by_action[action_index]))
        print(
            f"  {action_name}: {count} selections "
            f"(mean immediate reward: {average_reward:+.2f})"
        )

    print("\nAction-type totals:")
    print(f"  Task slots: {task_actions} ({task_actions / num_steps * 100:.1f}%)")
    print(f"  Charging: {charge_actions} ({charge_actions / num_steps * 100:.1f}%)")
    print(f"  Idle: {idle_actions} ({idle_actions / num_steps * 100:.1f}%)")

    categories = ["Task", "Charging", "Idle"]
    category_rewards = {name: [] for name in categories}
    for action_index, rewards in reward_by_action.items():
        _, action_type = env.decode_action(action_index)
        if action_type < env.max_tasks:
            category_rewards["Task"].extend(rewards)
        elif action_type < env.max_tasks + env.num_stations:
            category_rewards["Charging"].extend(rewards)
        else:
            category_rewards["Idle"].extend(rewards)

    mean_rewards = [
        float(np.mean(category_rewards[name])) if category_rewards[name] else 0.0
        for name in categories
    ]
    counts_by_category = [task_actions, charge_actions, idle_actions]

    fig, (action_axis, reward_axis) = plt.subplots(1, 2, figsize=(13, 5))
    colors = ["lightblue", "lightgreen", "lightcoral"]
    nonzero = [index for index, count in enumerate(counts_by_category) if count > 0]
    if nonzero:
        action_axis.pie(
            [counts_by_category[index] for index in nonzero],
            labels=[categories[index] for index in nonzero],
            colors=[colors[index] for index in nonzero],
            autopct="%1.1f%%",
            startangle=90,
        )
        action_axis.set_title("Selected action types")

    bars = reward_axis.bar(categories, mean_rewards, color=colors)
    reward_axis.set_ylabel("Mean immediate reward (model units)")
    reward_axis.set_title("Observed immediate reward by action type")
    reward_axis.axhline(0.0, color="black", linewidth=0.8)
    reward_axis.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, mean_rewards):
        vertical_alignment = "bottom" if value >= 0 else "top"
        offset = 3 if value >= 0 else -3
        reward_axis.annotate(
            f"{value:+.2f}",
            (bar.get_x() + bar.get_width() / 2, value),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            va=vertical_alignment,
            fontweight="bold",
        )

    fig.tight_layout()
    plt.show()
    return action_counts, reward_by_action


action_counts, reward_by_action = analyze_learned_policy(
    agent, env, num_steps=50
)

## Summary and interpretation

### What the notebook demonstrates

- The simulation turns dispatch decisions into state transitions and rewards, providing experience for a reinforcement-learning agent.
- A DQN estimates action values with a neural network. Epsilon-greedy action selection supports exploration, replay reuses stored transitions, and a periodically updated target network stabilizes the temporal-difference target.
- The state and action encodings have fixed dimensions. Fixed action slots keep each task, charging, and idle choice tied to a stable index even as the pending-task list changes.
- Evaluation, comparison, plots, and action analysis provide different views of behavior. A single metric or plot is not sufficient evidence of convergence or policy quality.

### Reading the results responsibly

Episode reward combines the modeled rewards and penalties. The task-completion fraction is completed tasks divided by the configured task target; the episode can end before reaching that target. Battery-violation counts track cases in which modeled task completion takes battery below the minimum, after which the simulation clamps the level to that minimum. In this environment, feasibility checks are intended to make such violations uncommon, but the count remains useful as a diagnostic.

Training reward and loss can fluctuate because the environment is stochastic and the policy changes during learning. A decreasing loss does not necessarily imply better dispatch performance, and a rising completion fraction over a limited run does not prove convergence. Test averages summarize only the tested episodes. More robust empirical conclusions would require broader evaluation across scenarios and repeated runs, which are outside this example's implemented scope.

### Strengths and limitations of the method

DQN can represent nonlinear action values and can learn from sequential experience in a dynamic setting. It also introduces sensitivity to reward design, state representation, network and learning settings, and training experience. The learned policy is not guaranteed to be optimal, and its decisions can be difficult to explain. The environment here omits route planning, detailed time windows, charger queues, and realistic travel and service dynamics, so its learned behavior should not be treated as an operational recommendation.

The next tier's digital-twin setting can provide a richer simulation context for evaluating policies under more realistic scenarios. Such extensions should preserve the same DQN solution approach when the goal is to study this method, while making the added assumptions and evaluation evidence explicit.